In [1]:
#### in this script, calculate rolling averages/merging model and observations ####

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import boto3

In [2]:
nbm_s3_url = "s3://processed-data-809918852303-us-east-1-an/nbm_maxt_final_2020_2026_7days.parquet"
nbm_df = pd.read_parquet(nbm_s3_url)

In [3]:
nbm_df['date'] = pd.to_datetime(nbm_df['date'])
nbm_df = nbm_df.sort_values(by=['public_zone', 'date', 'week', 'forecast_day'])

In [4]:
prism_s3_url = 's3://processed-data-809918852303-us-east-1-an/prism_maxt_final_2020_2026.parquet'
prism_df = pd.read_parquet(prism_s3_url)

In [5]:
# make sure rolling doesn't include current day!
prism_df['60daywindow_prism'] = (
    prism_df.groupby('unique_zone_str')['maxt_value']
    .transform(lambda x: x.shift(1).rolling(window=60, min_periods=60).mean())
)
prism_df['7daywindow_prism'] = (
    prism_df.groupby('unique_zone_str')['maxt_value']
    .transform(lambda x: x.shift(1).rolling(window=7, min_periods=7).mean())
)

In [11]:
# merge prism_df and nbm_df on date and zone string to ensure perfect matching
merged_df = pd.merge(
    nbm_df, 
    prism_df, 
    on=['date', 'unique_zone_str'], 
    suffixes=('_nbm', '_prism')
)

In [12]:
merged_df

,public_zone,date,week_nbm,maxt_value_nbm,forecast_day,unique_zone_str,model_run_date,zone_id_nbm,state_nbm,name_nbm,global_zone_id,maxt_value_prism,week_prism,zone_id_prism,state_prism,name_prism,60daywindow_prism,7daywindow_prism
0,1,2020-10-20,43,80.592466,1,AL_001,2020-10-20,001,AL,Lauderdale,1,80.438460,43,001,AL,Lauderdale,79.339964,73.473508
1,1,2020-10-21,43,81.503425,1,AL_001,2020-10-21,001,AL,Lauderdale,1,81.583053,43,001,AL,Lauderdale,79.380602,74.445798
2,1,2020-10-21,43,81.102740,2,AL_001,2020-10-20,001,AL,Lauderdale,1,81.583053,43,001,AL,Lauderdale,79.380602,74.445798
3,1,2020-10-22,43,81.886986,1,AL_001,2020-10-22,001,AL,Lauderdale,1,82.355829,43,001,AL,Lauderdale,79.372306,74.662086
4,1,2020-10-22,43,81.688356,2,AL_001,2020-10-21,001,AL,Lauderdale,1,82.355829,43,001,AL,Lauderdale,79.372306,74.662086
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52660295,3850,2026-07-17,29,95.678788,3,WY_199,2026-07-15,199,WY,Sheridan Foothills,3850,91.396461,29,199,WY,Sheridan Foothills,78.531191,97.448493
52660296,3850,2026-07-17,29,95.527273,4,WY_199,2026-07-14,199,WY,Sheridan Foothills,3850,91.396461,29,199,WY,Sheridan Foothills,78.531191,97.448493
52660297,3850,2026-07-17,29,94.435152,5,WY_199,2026-07-13,199,WY,Sheridan Foothills,3850,91.396461,29,199,WY,Sheridan Foothills,78.531191,97.448493
52660298,3850,2026-07-17,29,96.001212,6,WY_199,2026-07-12,199,WY,Sheridan Foothills,3850,91.396461,29,199,WY,Sheridan Foothills,78.531191,97.448493


In [13]:
merged_df = merged_df.drop(columns=['week_nbm', 'zone_id_nbm', 'state_nbm', 'name_nbm', 'zone_id_nbm', 'state_nbm', 'name_nbm', 'global_zone_id', 'unique_zone_str'])
merged_df = merged_df.rename(columns={'week_prism': 'week', 'state_prism': 'state', 'name_prism':'name', 'zone_id_prism':'zone_id'})

In [14]:
merged_df['nbm_minus_obs'] = merged_df['maxt_value_nbm'] - merged_df['maxt_value_prism']

In [15]:
column_order = [
        'public_zone', 'date', 'week', 'forecast_day', 'state', 'zone_id', 'name', 
        'model_run_date', 'maxt_value_nbm', 'maxt_value_prism', '60daywindow_prism', '7daywindow_prism','nbm_minus_obs'
]
merged_df = merged_df[column_order]

In [16]:
merged_df

,public_zone,date,week,forecast_day,state,zone_id,name,model_run_date,maxt_value_nbm,maxt_value_prism,60daywindow_prism,7daywindow_prism,nbm_minus_obs
0,1,2020-10-20,43,1,AL,001,Lauderdale,2020-10-20,80.592466,80.438460,79.339964,73.473508,0.154006
1,1,2020-10-21,43,1,AL,001,Lauderdale,2020-10-21,81.503425,81.583053,79.380602,74.445798,-0.079628
2,1,2020-10-21,43,2,AL,001,Lauderdale,2020-10-20,81.102740,81.583053,79.380602,74.445798,-0.480313
3,1,2020-10-22,43,1,AL,001,Lauderdale,2020-10-22,81.886986,82.355829,79.372306,74.662086,-0.468843
4,1,2020-10-22,43,2,AL,001,Lauderdale,2020-10-21,81.688356,82.355829,79.372306,74.662086,-0.667473
...,...,...,...,...,...,...,...,...,...,...,...,...,...
52660295,3850,2026-07-17,29,3,WY,199,Sheridan Foothills,2026-07-15,95.678788,91.396461,78.531191,97.448493,4.282327
52660296,3850,2026-07-17,29,4,WY,199,Sheridan Foothills,2026-07-14,95.527273,91.396461,78.531191,97.448493,4.130812
52660297,3850,2026-07-17,29,5,WY,199,Sheridan Foothills,2026-07-13,94.435152,91.396461,78.531191,97.448493,3.038690
52660298,3850,2026-07-17,29,6,WY,199,Sheridan Foothills,2026-07-12,96.001212,91.396461,78.531191,97.448493,4.604751


In [18]:
merged_df['nbm_minus_obs_window_60d'] = merged_df['maxt_value_nbm'] - merged_df['60daywindow_prism']
merged_df['nbm_minus_obs_window_7d'] = merged_df['maxt_value_nbm'] - merged_df['7daywindow_prism']

In [19]:
merged_df

,public_zone,date,week,forecast_day,state,zone_id,name,model_run_date,maxt_value_nbm,maxt_value_prism,60daywindow_prism,7daywindow_prism,nbm_minus_obs,nbm_minus_obs_window_60d,nbm_minus_obs_window_7d
0,1,2020-10-20,43,1,AL,001,Lauderdale,2020-10-20,80.592466,80.438460,79.339964,73.473508,0.154006,1.252502,7.118957
1,1,2020-10-21,43,1,AL,001,Lauderdale,2020-10-21,81.503425,81.583053,79.380602,74.445798,-0.079628,2.122823,7.057627
2,1,2020-10-21,43,2,AL,001,Lauderdale,2020-10-20,81.102740,81.583053,79.380602,74.445798,-0.480313,1.722138,6.656942
3,1,2020-10-22,43,1,AL,001,Lauderdale,2020-10-22,81.886986,82.355829,79.372306,74.662086,-0.468843,2.514681,7.224900
4,1,2020-10-22,43,2,AL,001,Lauderdale,2020-10-21,81.688356,82.355829,79.372306,74.662086,-0.667473,2.316051,7.026270
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52660295,3850,2026-07-17,29,3,WY,199,Sheridan Foothills,2026-07-15,95.678788,91.396461,78.531191,97.448493,4.282327,17.147596,-1.769705
52660296,3850,2026-07-17,29,4,WY,199,Sheridan Foothills,2026-07-14,95.527273,91.396461,78.531191,97.448493,4.130812,16.996081,-1.921220
52660297,3850,2026-07-17,29,5,WY,199,Sheridan Foothills,2026-07-13,94.435152,91.396461,78.531191,97.448493,3.038690,15.903960,-3.013342
52660298,3850,2026-07-17,29,6,WY,199,Sheridan Foothills,2026-07-12,96.001212,91.396461,78.531191,97.448493,4.604751,17.470021,-1.447281


In [20]:
print(merged_df['nbm_minus_obs'].describe().round(2))
print(merged_df['nbm_minus_obs_window_60d'].describe().round(2))
print(merged_df['nbm_minus_obs_window_7d'].describe().round(2))

count    52660300.00
mean           -0.15
std             4.23
min           -38.55
25%            -2.42
50%            -0.31
75%             1.92
max            50.29
Name: nbm_minus_obs, dtype: float64
count    52660300.00
mean            0.17
std            11.82
min           -63.02
25%            -7.35
50%             0.86
75%             8.21
max            51.94
Name: nbm_minus_obs_window_60d, dtype: float64
count    52660300.00
mean           -0.09
std             7.95
min           -52.49
25%            -4.73
50%             0.32
75%             4.87
max            46.93
Name: nbm_minus_obs_window_7d, dtype: float64


In [21]:
BUCKET_NAME = 'processed-data-809918852303-us-east-1-an'

merged_df.to_parquet('maxt_merged_df_7days_FULL.parquet', index=False)
s3 = boto3.client('s3')
s3.upload_file('maxt_merged_df_7days_FULL.parquet', BUCKET_NAME, 'maxt_merged_df_7days_FULL.parquet')
print("Parquet complete /uploaded  now")

Parquet complete /uploaded  now


In [9]:
# # trying to plot PDF
# sns.set_theme(style="whitegrid")
# plt.figure(figsize=(8, 5))
# plt.xlim(-15, 15)
# plt.xticks(np.arange(-20, 20, 4)) 

# # get every curve in
# sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 1, 'nbm_minus_obs'], fill=True, color='red', bw_adjust=1, label='Day 1')
# sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 2, 'nbm_minus_obs'], fill=True, color='orange', bw_adjust=1, label='Day 2')
# sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 3, 'nbm_minus_obs'], fill=True, color='yellow', bw_adjust=1, label='Day 3')
# sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 4, 'nbm_minus_obs'], fill=True, color='green', bw_adjust=1, label='Day 4')
# sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 5, 'nbm_minus_obs'], fill=True, color='blue', bw_adjust=1, label='Day 5')
# sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 6, 'nbm_minus_obs'], fill=True, color='purple', bw_adjust=1, label='Day 6')
# sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 7, 'nbm_minus_obs'], fill=True, color='pink', bw_adjust=1, label='Day 7')

# plt.xlabel('($^{\circ}$F)')
# plt.ylabel('Probability Density')
# plt.title(f"PDF of Temperature Differences, Model - Obs")
# plt.legend(loc='upper right')

# #plt.savefig('pdf_plot.png')
# plt.show()

In [10]:
# #temp_thresholds = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0]
# #temp_thresholds = [-10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
# #temp_thresholds = [-10, 10, -9, 9, -8, 8, -7, 7, -6, 6, -5, 5, -4, 4, -3, 3, -2, 2, -1 , 1]
# temp_thresholds = [2, 5, 8]
# forecast_days = [1, 2, 3, 4, 5, 6, 7]


# for temp in temp_thresholds:
#     for day in forecast_days:
#         if temp > 0:
#             mask = (merged_df.loc[merged_df['forecast_day'] == day, 'nbm_minus_obs'] >= temp)
#             column_name = 'outside_' + str(temp) + 'F'
#             merged_df[column_name] = mask
#             print(f"forecast day {day} total count above {temp}F is {merged_df[column_name].sum()}")
#             print(f"forecast day {day} total percentage above {temp}F is {(merged_df[column_name].sum()/len(merged_df))*100:.3f}%\n")
#         else:
#             mask = (merged_df.loc[merged_df['forecast_day'] == day, 'nbm_minus_obs'] <= temp)
#             column_name = 'outside_' + str(temp) + 'F'
#             merged_df[column_name] = mask
#             print(f"forecast day {day} total count below {temp}F is {merged_df[column_name].sum()}")
#             print(f"forecast day {day} total percentage below {temp}F is {(merged_df[column_name].sum()/len(merged_df))*100:.3f}%\n")

In [11]:
# temp_thresholds = [2, 5, 8]
# forecast_days = [1, 2, 3, 4, 5, 6, 7]


# for temp in temp_thresholds:
#     for day in forecast_days:
#         mask = (merged_df.loc[merged_df['forecast_day'] == day, 'nbm_minus_obs'] >= temp) | (merged_df.loc[merged_df['forecast_day'] == day, 'nbm_minus_obs'] <= -temp)
#         column_name = 'outside_' + str(temp) + 'F'
#         merged_df[column_name] = mask
#         print(f"forecast day {day} total count outside {temp}F is {merged_df[column_name].sum()}")
#         print(f"forecast day {day} total percentage outside {temp}F is {(merged_df[column_name].sum()/len(merged_df))*100:.3f}%\n")

In [12]:
merged_df['nbm_minus_obs_window'] = merged_df['max_temp_f_nbm'] - merged_df['60daywindow_prism']

column_order = [
        'public_zone', 'date', 'week', 'forecast_day', 'nbm_minus_obs', 'nbm_minus_obs_window', 'state', 'zone_id', 'name', 
        'model_run_date', 'max_temp_f_nbm', 'max_temp_f_prism', '60daywindow_prism'
]
merged_df = merged_df[column_order]

In [13]:
#create boolean masks for differences outside of 2°F, 5°F, and 8°F 
mask_2F = (merged_df['nbm_minus_obs'] >= 2) | (merged_df['nbm_minus_obs'] <= -2)
merged_df['outside_2F'] = mask_2F

mask_5F = (merged_df['nbm_minus_obs'] >= 5) | (merged_df['nbm_minus_obs'] <= -5)
merged_df['outside_5F'] = mask_5F

mask_8F = (merged_df['nbm_minus_obs'] >= 8) | (merged_df['nbm_minus_obs'] <= -8)
merged_df['outside_8F'] = mask_8F

In [14]:
print(f"total count outside of ±2F is {merged_df['outside_2F'].sum()}")
print(f"total count outside of ±5F is {merged_df['outside_5F'].sum()}")
print(f"total count outside of ±8F is {merged_df['outside_8F'].sum()}\n")

print(f"total percentage outside of ±2F is {(merged_df['outside_2F'].sum()/len(merged_df))*100:.3f}%")
print(f"total percentage outside of ±5F is {(merged_df['outside_5F'].sum()/len(merged_df))*100:.3f}%")
print(f"total percentage outside of ±8F is {(merged_df['outside_8F'].sum()/len(merged_df))*100:.3f}%")

total count outside of ±2F is 27137814
total count outside of ±5F is 9342687
total count outside of ±8F is 3344939

total percentage outside of ±2F is 53.676%
total percentage outside of ±5F is 18.479%
total percentage outside of ±8F is 6.616%


In [15]:
#create boolean masks for differences outside of 2°F, 5°F, and 8°F 
mask_2F = (merged_df['nbm_minus_obs_window'] >= 2) | (merged_df['nbm_minus_obs_window'] <= -2)
merged_df['outside_2F_window'] = mask_2F

mask_5F = (merged_df['nbm_minus_obs_window'] >= 5) | (merged_df['nbm_minus_obs_window'] <= -5)
merged_df['outside_5F_window'] = mask_5F

mask_8F = (merged_df['nbm_minus_obs_window'] >= 8) | (merged_df['nbm_minus_obs_window'] <= -8)
merged_df['outside_8F_window'] = mask_8F

print(f"total count window outside of ±2F is {merged_df['outside_2F_window'].sum()}")
print(f"total count window outside of ±5F is {merged_df['outside_5F_window'].sum()}")
print(f"total count window outside of ±8F is {merged_df['outside_8F_window'].sum()}\n")

print(f"total percentage outside window of ±2F is {(merged_df['outside_2F_window'].sum()/len(merged_df))*100:.3f}%")
print(f"total percentage outside window of ±5F is {(merged_df['outside_5F_window'].sum()/len(merged_df))*100:.3f}%")
print(f"total percentage outside window of ±8F is {(merged_df['outside_8F_window'].sum()/len(merged_df))*100:.3f}%")

total count window outside of ±2F is 38081987
total count window outside of ±5F is 23828804
total count window outside of ±8F is 14950712

total percentage outside window of ±2F is 75.323%
total percentage outside window of ±5F is 47.131%
total percentage outside window of ±8F is 29.571%


In [ ]:
merged_df = merged_df[merged_df['date'] >= '2020-10-20']
merged_df.reset_index(drop=True)

In [ ]:
# BUCKET_NAME = 'processed-data-809918852303-us-east-1-an'

# merged_df.to_csv('maxt_merged_df_7days_FULL.csv', index=False)
# s3 = boto3.client('s3')
# s3.upload_file('merged_df_7days_maxt.csv', BUCKET_NAME, 'maxt_merged_df_7days_FULL.csv')
# print("CSV complete /uploaded  now")

In [ ]:
merged_df

In [ ]:
# simple analysis
mae = merged_df['nbm_minus_obs'].abs().mean()
mbe = merged_df['nbm_minus_obs'].mean()

print(f"mean absolute error: {mae:.3f}°F")
print(f"mean bias wrror:     {mbe:.3f}°F")

In [ ]:
# simple analysis
mae = merged_df['nbm_minus_obs_window'].abs().mean()
mbe = merged_df['nbm_minus_obs_window'].mean()

print(f"mean absolute error: {mae:.3f}°F")
print(f"mean bias wrror:     {mbe:.3f}°F")

In [ ]:
# group by the day of week and calculate the mean for biases column
daily_averages = merged_df.groupby('forecast_day')['nbm_minus_obs'].describe()

print(daily_averages)

In [ ]:
# trying to plot PDF
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 5))
plt.xlim(-15, 15)
plt.xticks(np.arange(-20, 24, 4)) 

# get every curve in
sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 1, 'nbm_minus_obs_window'], fill=False, color='red', bw_adjust=1, label='Day 1')
sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 2, 'nbm_minus_obs_window'], fill=False, color='orange', bw_adjust=1, label='Day 2')
sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 3, 'nbm_minus_obs_window'], fill=False, color='yellow', bw_adjust=1, label='Day 3')
sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 4, 'nbm_minus_obs_window'], fill=False, color='green', bw_adjust=1, label='Day 4')
sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 5, 'nbm_minus_obs_window'], fill=False, color='blue', bw_adjust=1, label='Day 5')
sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 6, 'nbm_minus_obs_window'], fill=False, color='purple', bw_adjust=1, label='Day 6')
sns.kdeplot(data=merged_df, x=merged_df.loc[merged_df['forecast_day'] == 7, 'nbm_minus_obs_window'], fill=False, color='pink', bw_adjust=1, label='Day 7')

plt.xlabel('($^{\circ}$F)')
plt.ylabel('Probability Density')
plt.title(f"PDF of Temperature Differences, Model - Obs 60 Day Window")
plt.legend(loc='upper right')

#plt.savefig('pdf_plot_window_zoomed.png')
plt.show()

In [ ]:
# attempting to plot NBM biases

plot_day = '2022-01-06'
day_diff_df = merged_df[merged_df['date'] == plot_day]

zones_gdf = gpd.read_file("https://www.weather.gov/source/gis/Shapefiles/WSOM/z_16ap26.zip")
zones_gdf['unique_zone_str'] = zones_gdf['STATE'] + '_' + zones_gdf['ZONE']
zones_gdf['STATE_ZONE'] = zones_gdf['STATE'] + '_' + zones_gdf['ZONE']

zones_display = zones_gdf.to_crs(epsg=4269)
map_gdf = zones_display.merge(day_diff_df, left_on='STATE_ZONE', right_on='unique_zone_str')

#initialize the plot figure
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
zones_display.plot(ax=ax, color='lightgrey', edgecolor='none')
plt.grid(linestyle='--', color='grey', alpha=0.2)

plt.xlim(-126, -65)
plt.ylim(24, 50)

max_error_bound = 10

map_gdf.plot(
    column='temp_diff_f',
    ax=ax,
    legend=True,
    legend_kwds={
        'label': 'temperature diff in °F', 
        'orientation': 'horizontal', 
        'extend': 'both',
        'pad': 0.05
    },
    cmap='RdBu_r',
    vmin=-max_error_bound,
    vmax=max_error_bound
)

plt.title('NBM bias for ' + plot_day + ' forecast day 1', fontsize=13, fontweight='bold') # this is model - obs
plt.tight_layout()
plt.show()